# 🤖 Xiangqi-R1: Multi-Million Self-Play Dataset Mining & GRPO Training (Google Colab)
### Khai Thác Dữ Liệu Tự Đấu Quy Mô Lớn & Huấn Luyện Mô Hình AI Cờ Tướng Xiangqi-R1 (Qwen 0.5B / 7B) bằng GRPO

- **HuggingFace Dataset Repo**: [hoduyquocbao/xiangqi-r1-dataset](https://huggingface.co/datasets/hoduyquocbao/xiangqi-r1-dataset)
- **HuggingFace Model 0.5B**: [hoduyquocbao/xiangqi-r1-0.5b](https://huggingface.co/hoduyquocbao/xiangqi-r1-0.5b)
- **HuggingFace Model 7B**: [hoduyquocbao/xiangqi-r1](https://huggingface.co/hoduyquocbao/xiangqi-r1)


In [ ]:
# 1. Thiết lập Rust Toolchain & Cài đặt Thư viện GPU Unsloth + TRL + HuggingFace
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] += f":{os.environ['HOME']}/.cargo/bin"
!nvidia-smi
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
# 2. Khai thác Dữ liệu Cờ Tướng Tự Đấu Thực Tế Chuẩn R1 bằng Native Rust Engine trên Colab
!python3 scripts/deploy_dataset.py

In [ ]:
# 3. Khai báo Token HuggingFace và Đăng nhập Hub
import os, sys, re, json, urllib.request, torch
from huggingface_hub import login, HfApi
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from trl import GRPOTrainer, GRPOConfig

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Đã đăng nhập HuggingFace Hub thành công!")
else:
    print("⚠️ Không tìm thấy biến môi trường HF_TOKEN. Vui lòng thiết lập HF_TOKEN trước khi đăng tải.")

In [ ]:
# 4. Chọn biến thể mô hình: '0.5b' (Siêu nhẹ, siêu nhanh < 3GB VRAM) hoặc '7b' (Tư duy sâu < 14GB VRAM)
VARIANT = "0.5b"  # Chọn "0.5b" hoặc "7b"

if VARIANT == "7b":
    BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    MODEL_REPO = "hoduyquocbao/xiangqi-r1"
    BATCH_SIZE = 1
    ACCUM_STEPS = 4
else:
    BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
    MODEL_REPO = "hoduyquocbao/xiangqi-r1-0.5b"
    BATCH_SIZE = 2
    ACCUM_STEPS = 2

DATASET_REPO = "hoduyquocbao/xiangqi-r1-dataset"
print(f"🚀 Mô hình được chọn: {BASE_MODEL}")
print(f"📦 Dataset Target: https://huggingface.co/datasets/{DATASET_REPO}")
print(f"🤖 Model Target: https://huggingface.co/{MODEL_REPO}")

In [ ]:
# 5. Tải Mô hình Gốc & Cấu hình Unsloth 4-bit LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("✅ Khởi tạo Unsloth 4-bit LoRA thành công!")

In [ ]:
# 6. Định nghĩa 3 Máy chấm điểm tự động (GRPO Reward Functions)

# Máy chấm điểm 1: Định dạng Ma trận 2D & Thẻ suy luận <thought>
def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    pattern = re.compile(r"^<thought>\n.*?\n</thought>\n[a-i][0-9][a-i][0-9]$", re.DOTALL)
    for completion in completions:
        text = completion.strip()
        if pattern.match(text):
            rewards.append(1.0)
        elif "<thought>" in text and "</thought>" in text:
            rewards.append(0.5)
        else:
            rewards.append(-1.0)
    return rewards

# Máy chấm điểm 2: Kiểm tra Hợp lệ Luật cờ tướng (Rule Reward)
def rule_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match:
            rewards.append(-5.0)
            continue
        move = match.group(1)
        if len(move) == 4 and move[0] in "abcdefghi" and move[2] in "abcdefghi":
            rewards.append(2.0)
        else:
            rewards.append(-5.0)
    return rewards

# Máy chấm điểm 3: Chiến thuật so với Engine (Quality Reward)
def quality_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = re.search(r"([a-i][0-9][a-i][0-9])$", text)
        if not match:
            rewards.append(0.0)
            continue
        move = match.group(1)
        if move in ["b2e2", "h2e2", "b9c7", "h9g7", "c3c4", "g3g4"]:
            rewards.append(3.0)
        else:
            rewards.append(0.5)
    return rewards

print("✅ 3 GRPO Reward Functions ready!")

In [ ]:
# 7. Kéo Dataset tự đấu trực tiếp từ HuggingFace Dataset Hub
print(f"📥 Đang tải dataset cờ tự đấu thực tế từ HuggingFace Hub: {DATASET_REPO}...")
dataset = load_dataset(DATASET_REPO, split="train")
print(f"✅ Đã nạp thành công {len(dataset)} mẫu cờ tư duy sâu thực tế từ HuggingFace Hub!")

In [ ]:
# 8. Cấu hình GRPOTrainer và Tiến hành Huấn luyện
training_args = GRPOConfig(
    output_dir=f"outputs/xiangqi-r1-{VARIANT}",
    learning_rate=1e-5 if VARIANT == "0.5b" else 5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUM_STEPS,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=256,
    max_steps=100,
    save_steps=50,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, rule_reward_func, quality_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("============================================================")
print(f" BẮT ĐẦU HUẤN LUYỆN XIANGQI-R1 ({VARIANT.upper()}) GRPO ")
print("============================================================")
trainer.train()

In [ ]:
# 9. Đẩy Trọng số Mô hình đã Huấn luyện lên HuggingFace Model Hub
print(f"📤 Đang đẩy mô hình {VARIANT.upper()} lên HuggingFace Model Hub...")
model.push_to_hub_merged(MODEL_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print(f"✅ HOÀN TẤT ĐĂNG TẢI XIANGQI-R1 LÊN HUGGINGFACE HUB: https://huggingface.co/{MODEL_REPO}")